In [16]:
import reframed
import pandas as pd
from pathlib import Path


import re
import requests
from bs4 import BeautifulSoup



In [3]:
def extract_charge(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Convert visible text to a single string
    text = " ".join(soup.stripped_strings)

    # Look for: "Charges in BiGG models: -2"
    match = re.search(r"Charges in BiGG models:\s*([-+]?\d+)", text)
    if not match:
        raise ValueError("Charge not found on page")

    return match.group(1)


In [5]:
repo_folder = Path("../..")
data_folder = repo_folder / "data" / "7_GEM_reconstruction"
# gapfilled_folder = data_folder / 'models/gapfilled_FBA'
# polished_folder = data_folder / 'models/polished'
bigg_universe_fn = data_folder / 'carveme_universe_bacteria_fixed.xml'

In [6]:
universe = reframed.load_cbmodel(bigg_universe_fn)

In [8]:
missing_charge = []
for m in universe.metabolites.values():
    try:
        m.metadata['CHARGE']
    except KeyError:
        print(f'Metabolite {m.id} is missing charge annotation.')
        missing_charge.append(m.id[2:-2])

Metabolite M_12dag3p_BS_c is missing charge annotation.
Metabolite M_12dgr_BS_c is missing charge annotation.
Metabolite M_12dgr_LLA_c is missing charge annotation.
Metabolite M_14dh2napcoa_c is missing charge annotation.
Metabolite M_1ag160_e is missing charge annotation.
Metabolite M_1ag180_e is missing charge annotation.
Metabolite M_1ag181d9_e is missing charge annotation.
Metabolite M_1ag182d9d12_e is missing charge annotation.
Metabolite M_1hdec9eg3p_c is missing charge annotation.
Metabolite M_1hdec9eg3p_p is missing charge annotation.
Metabolite M_1py4h3c_c is missing charge annotation.
Metabolite M_23dhacoa_c is missing charge annotation.
Metabolite M_23dhbzs2_c is missing charge annotation.
Metabolite M_23dhbzs3_c is missing charge annotation.
Metabolite M_23dhbzs3_e is missing charge annotation.
Metabolite M_23dhbzs3_p is missing charge annotation.
Metabolite M_24dhhed_c is missing charge annotation.
Metabolite M_26dapime_c is missing charge annotation.
Metabolite M_2a3o4pob

In [9]:
missing_charge = list(set(missing_charge))

In [10]:
len(missing_charge)

535

In [14]:
bigg_url = 'http://bigg.ucsd.edu/universal/metabolites/'
charge_data = []
for met in missing_charge:
    url = bigg_url + met
    try:
        charge = extract_charge(url)
        print(f'Metabolite {met} has charge {charge}.')
    except Exception as e:
        print(f'Could not retrieve charge for metabolite {met}: {e}')
        charge = 'NA'
    charge_data.append([met, charge])


Metabolite 2ptcoa has charge -4.
Metabolite osuc has charge -3.
Metabolite 3o6athcoa has charge -4.
Metabolite 2hh24dd has charge -2.
Could not retrieve charge for metabolite acysbmn: Charge not found on page
Metabolite prealginate_G has charge -5.
Metabolite 2ddecg3p has charge -1.
Metabolite m2butp has charge 0.
Metabolite 4abzglu has charge -2.
Metabolite 3ophxacoa has charge -4.
Metabolite S2hglut has charge -2.
Metabolite prealg_MG_14 has charge -5.
Metabolite 3hdd6coa has charge -4.
Metabolite 2agpe161 has charge 0.
Metabolite alltn__R has charge 0.
Metabolite gccoa has charge 0.
Metabolite alatrp has charge 0.
Metabolite 3otdccoa has charge -4.
Metabolite ala_L_glu__L has charge -1.
Metabolite 3mba has charge -1.
Metabolite hdd4coa has charge -4.
Metabolite ib_p has charge -1.
Metabolite al26da has charge -2.
Metabolite alahis has charge 0.
Could not retrieve charge for metabolite glucan4: Charge not found on page
Metabolite omaketo has charge -3.
Metabolite gg15dap has charge 1

In [17]:
charge_df = pd.DataFrame(charge_data, columns=['metabolite_id', 'charge'])
charge_df.to_csv(data_folder / f'missing_metabolite_charges_bigg.csv', index=False)